# Day 5 — AI Harness and Automation

## Daily project: Mini AI Harness + Website Maintenance Agent

This is the classroom master notebook for Day 5. Work from top to bottom: each section introduces one limitation, adds one system layer, and carries that improvement into the daily project.

### How to use this notebook

- Run the **Environment setup** cell directly below first. On Google Colab it clones the repository, installs packages, and asks for your API key. On your own computer it only loads the `.env` file.
- Every section starts with a small setup cell of its own; if the kernel restarts, rerun that cell and continue.
- Run the code cells in order and read the printed output: each cell prints what changed and why.
- Every lesson ends with a short **Checkpoint** (answers are folded under *Show answer*) and a **Recap**.
- Without an API key everything runs in deterministic **mock** mode and spends no credit. Use the instructor-issued OpenRouter key only for the marked live observations.
- Section 5.8 is the day's single hands-on exercise; a commented reference solution follows its check.

### Day 5 contents

1. [What Is an AI Harness?](#day-5-section-1)
2. [Model Configuration and Runtime](#day-5-section-2)
3. [Tool Registry and Discovery](#day-5-section-3)
4. [Permissions, Approval and Limits](#day-5-section-4)
5. [Events, Logs and Checkpoints](#day-5-section-5)
6. [MCP Client: Discover, Then Govern](#day-5-section-6)
7. [Day 5 Project — Mini AI Harness](#day-5-section-7)
8. [Pivotal Exercise: Build a Capability-Aware Tool Registry](#day-5-section-8)
9. [Day 5 Capstone — Website Maintenance Agent](#day-5-section-9)

---


In [ ]:
# --- Environment setup: run this cell first (Colab or local) -------------------------
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/cto-school/agentic-ai-engineering.git"   # the public course repository
REPO_DIR = Path("/content/agentic-ai-engineering")

if IN_COLAB:
    if not REPO_DIR.exists():
        print("Cloning the course repository ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
        print("Installing requirements (this takes a minute) ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "setup/requirements-core.txt")], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "setup/requirements-day5.txt")], check=True)
    os.chdir(REPO_DIR / "day_05_ai_harness")
    # Colab has no .env file. Paste the key you were issued; it is kept only in this runtime.
    from getpass import getpass
    if not os.getenv("OPENROUTER_API_KEY"):
        key = getpass("OPENROUTER_API_KEY (press Enter to stay in mock mode): ").strip()
        if key:
            os.environ["OPENROUTER_API_KEY"] = key
else:
    # Local machine: the key is read from the .env file at the repository root
    # (Day 1.1 explains how to create it from .env.example).
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))

print("Working directory:", os.getcwd())
print("Mode:", "LIVE (OpenRouter key found)" if os.getenv("OPENROUTER_API_KEY") else "MOCK (no key found: deterministic answers, no credit spent)")


<a id="day-5-section-1"></a>

## 5.1 — What Is an AI Harness?

For four days we built one agent at a time. Each project quietly repeated the same
chores: pick a model, describe the tools, run a loop, check whether an action was
allowed, remember something, print what happened.

A **runtime** is the code that executes one run. A **harness** is the runtime plus
everything around it — configuration, tool registry, policy, events, checkpoints.
This lesson names those repeated chores and shows you where each one now lives.


## Before you begin

### Learning outcomes

- Name the responsibilities every Day 1-4 project repeated.
- Point at the mini-harness module that now owns each one.
- Tell an agent, a framework, a protocol, a runtime and a harness apart.

Architecture reference: [Day 5 diagrams D16](../diagrams/source/day_05.md).

### Expected observation

Two different agent configurations print different behaviour while pointing at the same runtime modules. A printed table links each Day 1-4 pain point to one component.


## Concept briefing

## Why consolidate the earlier projects

By Day 5, several applications repeat the same responsibilities: load model
configuration, describe tools, validate arguments, enforce policy, limit steps, record
events and save continuation state. Copying this code into every agent makes safety fixes
inconsistent. A reusable runtime centralises the execution lifecycle.

This course uses **harness** as an umbrella term for the environment around an agent. In
industry, related terms include agent runtime, orchestration layer and agent platform.
The exact vocabulary varies; the responsibilities are transferable.


### Before you run: the `.env` file

Day 5 runs end to end with **no API key**. If you want the optional live cells, the key
lives in a file called `.env` at the repository root (next to `README.md`), containing
`OPENROUTER_API_KEY=sk-or-...` and `OPENROUTER_MODEL=openai/gpt-oss-120b`.
**Day 1.1 walks through creating it.** Without it the setup cell below simply reports
`MOCK` and every lesson still works.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — Two agents, described as data

An agent configuration is a JSON file, not code. It says who the agent is, which tools
it may see, and how many steps it gets. Load two of them and look at the difference.


In [ ]:
# Read both configurations from configs/*.json.
research = load_config("research_agent")
task = load_config("task_agent")

for config in (research, task):
    print("Agent name    :", config.name)
    print("  instructions:", config.instructions)
    print("  allowed tools:", config.allowed_tools)   # the allow-list, not "all tools"
    print("  max steps    :", config.max_steps)
    print("  model        :", config.model.provider, "/", config.model.model)
    print()

print("Two different agents. Zero lines of agent-specific Python so far.")

## Step 2 — The problem each harness component solves

Every component of the mini harness exists because something hurt on an earlier day.
This table is the whole point of Day 5, so we print it rather than describing it.


In [ ]:
# Left column: something that was awkward or unsafe in an earlier project.
# Right column: the mini-harness module that now owns it, once, for every agent.
LINEAGE = [
    ("Day 1", "every script rebuilt the API request by hand",
     "schemas.ModelConfig + providers.build_provider"),
    ("Day 1", "tools were if/elif branches inside the loop",
     "registry.ToolRegistry"),
    ("Day 2", "each project re-invented where retrieved state lived",
     "events.EventStore + events.JSONCheckpointStore"),
    ("Day 3", "safety checks were copy-pasted into each agent",
     "policy.decide"),
    ("Day 3", "approval meant 'ask in the chat and hope'",
     "runtime pause + checkpoint + runtime.resume"),
    ("Day 4", "a chatty team could loop and nobody could stop it",
     "runtime.MAX_STEPS_HARD_CAP"),
    ("Day 4", "explaining what happened meant re-reading print statements",
     "events.EventStore"),
]

width = max(len(problem) for _, problem, _ in LINEAGE)
print(f"{'Day':<6} {'Problem we actually hit':<{width}}  ->  Harness component")
print("-" * (width + 50))
for day, problem, component in LINEAGE:
    print(f"{day:<6} {problem:<{width}}  ->  {component}")

## Step 3 — The modules, and what each one refuses to do

Look at the package itself. Each module is small on purpose: you should be able to read
any one of them in a couple of minutes.


In [ ]:
import mini_harness

RESPONSIBILITIES = {
    "schemas":   "the data contracts (config, tool spec, decision, result)",
    "providers": "talk to a model; mock, OpenRouter or Ollama",
    "registry":  "hold tools, describe them, validate arguments",
    "policy":    "answer 'may this agent do this now?'",
    "runtime":   "run the loop, apply limits, pause for approval",
    "events":    "append-only record + resumable checkpoint",
    "memory":    "a deliberately tiny recall interface",
    "mcp_client": "discover capabilities from an outside server",
}
for module, job in RESPONSIBILITIES.items():
    print(f"mini_harness.{module:<11} : {job}")

print()
print("Public names the harness exports:", len(mini_harness.__all__))
print("Note what is NOT here: authentication, sandboxing, deployment, a UI.")
print("Those are real responsibilities - they are just not this course's subject.")

## Step 4 — Vocabulary, checked against the objects

These five words get used interchangeably online. Here they mean specific things, and
you can see each one as a Python object.


In [ ]:
from mini_harness import HarnessRuntime, MockModel, build_demo_registry

runtime = HarnessRuntime(build_demo_registry(), MockModel())

print("agent      :", research.name, "- a CONFIGURATION;", type(research).__name__)
print("framework  : the library you import; here, plain Python +", type(runtime).__name__)
print("protocol   : an interoperability contract, e.g. MCP (Day 5.6). Not shown yet.")
print("runtime    : the object that executes one run ->", type(runtime).__name__)
print("harness    : that runtime PLUS registry, policy, events, checkpoints")
print()
print("Same runtime object, two agents:")
print("  ", runtime.run(research, "What is a harness?").status,
      "<- research_agent")
print("  ", runtime.run(task, "Prepare a short update").status,
      "<- task_agent")

### Try it yourself

Before running the next cell, predict: a configuration asks for `max_steps = 500`.
How many steps will the runtime actually take?


In [ ]:
# --- Worked solution ---
# The configuration is a REQUEST. The runtime owns the ceiling, so the effective
# limit is min(config.max_steps, MAX_STEPS_HARD_CAP) - never the larger number.
from mini_harness import MAX_STEPS_HARD_CAP, effective_step_limit

greedy = load_config("research_agent")
greedy.max_steps = 500                      # a configuration can ask for anything

print("Hard cap written into runtime.py :", MAX_STEPS_HARD_CAP)
print("Steps this configuration requests:", greedy.max_steps)
print("Steps it will actually be given  :", effective_step_limit(greedy))
print()
print("A modest config is NOT raised to the cap:")
modest = load_config("research_agent")      # max_steps = 3 in the JSON file
print("  requested:", modest.max_steps, "-> effective:", effective_step_limit(modest))
print()
print("Lesson: limits belong to the runtime, not to the thing being limited.")

### Checkpoint

**1. What is the difference between an agent and a runtime?**

<details><summary>Show answer</summary>

An agent is a *configuration* - instructions, an allow-list of tools, limits, a model choice. It is data and lives in `configs/*.json`. A runtime is the *code* that executes that configuration. One runtime ran both agents in Step 4 without knowing anything about either of them.

</details>

**2. Why is `max_steps` in the agent config but the hard cap in `runtime.py`?**

<details><summary>Show answer</summary>

Because they answer different questions. `max_steps` is what this application thinks it needs; the hard cap is what the platform is willing to allow. If the ceiling lived in the config, any config could raise it, and the limit would stop being a limit.

</details>

### Recap

- Limitation: every Day 1-4 project re-implemented configuration, tools, policy, limits and logging, so a fix in one never reached the others.
- Layer added: a named module per responsibility, with agent behaviour pushed out into JSON configuration files.
- Evidence: one `HarnessRuntime` object ran two different agents, and a config asking for 500 steps was silently held to the runtime's cap of 10.


---

### Section 5.1 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-2"></a>

## 5.2 — Model Configuration and Runtime

Configuration is data; execution is code. If every agent builds its own HTTP request, then
every agent has its own bugs, its own timeout and its own way of hiding a cost.

Here one adapter hides the provider, and the runtime never learns which model it is using.


## Before you begin

### Learning outcomes

- Read a model configuration and build the matching provider adapter.
- Run the same agent through the mock provider (and OpenRouter, if a key is present).
- Read a run's event trace and find where usage and cost would be recorded.

Architecture reference: [Day 5 diagrams D16](../diagrams/source/day_05.md).

### Expected observation

Mock mode completes locally and reports empty usage. A mistyped provider name is rejected before any request is made.


## Concept briefing

## Configuration versus runtime

An agent configuration describes application-specific behavior: instructions, allowed
tools, model settings and limits. The runtime executes that configuration. A research
agent and a safe task agent should use one runtime without sharing inappropriate tools or
permissions.

A provider adapter hides API-specific request and response shapes behind a small
interface. Switching mock, OpenRouter or Ollama should not rewrite policy or the registry.
Provider metadata such as tokens, cost, latency and errors should still be preserved in
events.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — Configuration really is just a file

Nothing clever happens here. We print the raw JSON, then the dataclass it becomes.


In [ ]:
raw = (PROJECT_ROOT / "configs" / "research_agent.json").read_text(encoding="utf-8")
print("--- configs/research_agent.json ---")
print(raw)

research = load_config("research_agent")
print("--- as a Python object ---")
print("type          :", type(research).__name__)
print("model config  :", research.model)
print("temperature   :", research.model.temperature, "(0.0 = as repeatable as the model allows)")
print("max output    :", research.model.max_output_tokens, "tokens")

## Step 2 — One function chooses the adapter

`build_provider` is the only place that knows how each provider is reached. Notice that
we never write `os.environ["OPENROUTER_API_KEY"]`: a missing key means mock mode, not a crash.


In [ ]:
from mini_harness import build_provider

if LIVE:
    # A key is present, so point the SAME config at OpenRouter.
    research.model.provider = "openrouter"
    research.model.model = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

provider = build_provider(research.model)
print("Configured provider :", research.model.provider)
print("Adapter object      :", type(provider).__name__)
print("Model name          :", research.model.model)
print()
print("The runtime below is handed this object and never asks what it is.")

## Step 3 — Run it, and fall back if the network misbehaves

Every live call is wrapped. One 400 or one timeout must not stop the class, so we catch
the exception, explain it in one line, and rerun through the mock provider.


In [ ]:
from mini_harness import HarnessRuntime, MockModel, build_demo_registry

registry = build_demo_registry()
runtime = HarnessRuntime(registry, provider)

try:
    result = runtime.run(research, "What does a harness centralise?")
except Exception as exc:                      # noqa: BLE001 - teaching fallback
    print("Live call failed:", type(exc).__name__, exc)
    print("Falling back to the mock provider so the lesson continues.")
    research.model.provider = "mock"
    runtime = HarnessRuntime(registry, MockModel())
    result = runtime.run(research, "What does a harness centralise?")

print("Run id :", result.run_id)
print("Status :", result.status)
print("Output :", result.output)

## Step 4 — The event trace is the same shape either way

This is the payoff of the adapter: swapping providers changes the *content* of events,
never their *shape*. That is what makes runs comparable.


In [ ]:
for event in result.events:
    print(f"{event['event']:<18}", event["details"])

print()
usage = [e["details"]["usage"] for e in result.events if e["event"] == "model_completed"]
print("Usage records attached to model_completed events:", usage)
print("In MOCK mode usage is empty on purpose: nothing was bought, so nothing is counted.")
print("In LIVE mode each record carries prompt_tokens, completion_tokens and cost_usd,")
print("which is how you attribute cost to a whole run rather than to one reply.")

## Step 5 — What belongs where

Three settings, three different homes. Getting this wrong is the most common
configuration bug in agent projects.


In [ ]:
print("temperature, max_output_tokens -> ModelConfig  (how the model writes)")
print("   ->", research.model)
print()
print("max_steps, allowed_tools      -> AgentConfig  (what the agent may do)")
print("   -> max_steps:", research.max_steps, "| allowed_tools:", research.allowed_tools)
print()
print("OPENROUTER_API_KEY            -> the .env file (a secret; never JSON, never a notebook)")
print("   -> key present in this session:", LIVE)

### Try it yourself

Predict what happens if a configuration names a provider that does not exist —
does the harness guess, or does it stop?


In [ ]:
# --- Worked solution ---
# build_provider knows exactly three provider names. Anything else is a
# configuration error, and it is raised BEFORE a request is built or a key is read.
from mini_harness import ModelConfig, build_provider

typo = ModelConfig(provider="openrouetr", model="openai/gpt-oss-120b")  # deliberate typo
try:
    build_provider(typo)
except ValueError as exc:
    print("Rejected:", exc)
    print("-> failing here is good: a typo can never silently become 'no model at all'.")

# The registry and the policy never saw this. They are not part of the provider layer.
print()
print("Registry still holds:", [spec.name for spec in build_demo_registry().discover()])
print("Fix the spelling and the exact same runtime works again:")
print("  ", type(build_provider(ModelConfig(provider="mock"))).__name__)

### Checkpoint

**1. Why does `build_provider` exist instead of the runtime calling `urllib` directly?**

<details><summary>Show answer</summary>

So that the request format of one vendor cannot leak into the loop. The runtime only knows `provider.decide(...) -> ModelDecision`. Swapping mock, OpenRouter or Ollama changes one line of configuration and touches no policy, registry or loop code.

</details>

**2. In mock mode the usage records are empty. Is that a bug?**

<details><summary>Show answer</summary>

No. Mock mode makes no network call and buys no tokens, so there is genuinely nothing to report. The important part is that the *field* exists in every `model_completed` event, so the same reading code works for a live run where the numbers are real.

</details>

### Recap

- Limitation: agents that build their own API requests duplicate bugs, timeouts and cost handling.
- Layer added: a provider adapter behind one `decide()` contract, selected by data.
- Evidence: the same agent config ran through the chosen adapter and produced an identically shaped event trace; a mistyped provider name was refused up front.


---

### Section 5.2 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-3"></a>

## 5.3 — Tool Registry and Discovery

On Day 1 a tool was an `if name == "search":` branch. That works for one tool and one
agent. It stops working the moment two agents need *different* tools.

A registry stores each tool once — name, description, input schema, risk level and the
function — and hands out only the subset an agent is allowed to see.


## Before you begin

### Learning outcomes

- Register tools with a schema and a risk level, and list them.
- Give each agent a scoped view of the registry instead of everything.
- Reject malformed arguments before any function runs.

Architecture reference: [Day 5 diagrams D16](../diagrams/source/day_05.md).

### Expected observation

The research agent sees one tool while four are registered, and a draft with no body is refused before `create_draft` is ever called.


## Concept briefing

## Registry, validation and policy

A tool registry stores names, descriptions, input schemas, executors and local risk
classifications. Discovery answers "what capabilities are visible?" Validation answers
"are these arguments structurally acceptable?" Policy answers "may this agent execute
this action now?" These are separate decisions.

The runtime should fail closed on unknown tools, invalid arguments and disallowed actions.
It should never ask the same model that proposed an action to make the authoritative
permission decision.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — What a registered tool actually carries

Four tools, deliberately spanning the four risk levels you will meet in Day 5.4.


In [ ]:
from mini_harness import build_demo_registry

registry = build_demo_registry()

print(f"{'tool':<16}{'risk':<13}{'required arguments'}")
print("-" * 60)
for spec in registry.discover():
    required = ", ".join(spec.input_schema.get("required", [])) or "(none)"
    print(f"{spec.name:<16}{spec.risk:<13}{required}")

print()
print("Full description sent to the model for one tool:")
print("  name       :", registry.get("create_draft").spec.name)
print("  description:", registry.get("create_draft").spec.description)
print("  schema     :", registry.get("create_draft").spec.input_schema)

## Step 2 — Discovery is scoped per agent

`discover(allowed)` is what gets turned into the tool list sent to the model. A tool the
agent may not use is never mentioned to it at all.


In [ ]:
research = load_config("research_agent")
task = load_config("task_agent")

print("Registered in total     :", [s.name for s in registry.discover()])
print("research_agent can see  :", [s.name for s in registry.discover(research.allowed_tools)])
print("task_agent can see      :", [s.name for s in registry.discover(task.allowed_tools)])
print()
print("erase_workspace is registered but appears in neither list.")
print("Not mentioning a tool is the cheapest safety control there is.")

## Step 3 — A valid call

`call()` validates first, then runs the function. Here everything is in order.


In [ ]:
result = registry.call("lookup_notes", {"query": "harness"})
print("Tool     :", "lookup_notes")
print("Arguments:", {"query": "harness"})
print("Returned :", result)

## Step 4 — Four ways to be rejected

Validation is structural: it checks the *shape* of the arguments. Each failure below
happens before the tool function is entered, so no side effect can occur.


In [ ]:
attempts = [
    ("create_draft", {"subject": "Body is missing"},          "a required argument is absent"),
    ("create_draft", {"subject": "x", "body": 42},            "an argument has the wrong type"),
    ("lookup_notes", {"query": "ok", "extra": "surprise"},    "an argument nobody declared"),
    ("no_such_tool", {"query": "ok"},                         "the tool does not exist"),
]

for name, arguments, why in attempts:
    try:
        registry.call(name, arguments)
    except Exception as exc:                  # noqa: BLE001 - we want to show the type
        print(f"{why:<32} -> {type(exc).__name__}: {exc}")
    else:
        print(f"{why:<32} -> ACCEPTED (this would be a bug)")

print()
print("None of the four tool functions ran. That is the point of validating first.")

## Step 5 — Being visible is not being allowed

This is the sentence to remember from Day 5. `send_email` is in the task agent's
allow-list and passes validation — and still does not get to run on its own.


In [ ]:
from mini_harness import decide

spec = registry.get("send_email").spec
print("Is send_email discoverable by task_agent?",
      spec.name in [s.name for s in registry.discover(task.allowed_tools)])
print("Do these arguments validate?", end=" ")
registry.validate("send_email", {"to": "a@b.test", "subject": "s", "body": "b"})
print("yes")
print("So may it run?  ->", decide(task, spec))
print()
print("Three separate questions:")
print("  discovery  : does this agent get told the tool exists?")
print("  validation : are these arguments structurally acceptable?")
print("  policy     : may this agent perform this action right now?  (Day 5.4)")

### Try it yourself

Register your own read-only tool with one required string argument, then prove that
passing a number instead of a string is refused.


In [ ]:
# --- Worked solution ---
from mini_harness import ToolRegistry, ToolSpec

mine = ToolRegistry()

# A ToolSpec is: name, description, JSON-Schema for the arguments, local risk level.
mine.register(
    ToolSpec(
        name="word_count",
        description="Count the words in a piece of text",
        input_schema={"type": "object",
                      "properties": {"text": {"type": "string"}},
                      "required": ["text"],
                      "additionalProperties": False},   # refuse anything not declared
        risk="read",                                     # no side effect -> read
    ),
    lambda text: {"words": len(text.split())},           # the actual implementation
)

print("Registered:", [s.name for s in mine.discover()])
print("Valid call:", mine.call("word_count", {"text": "a harness is a runtime plus its rules"}))

try:
    mine.call("word_count", {"text": 12345})             # a number, not a string
except TypeError as exc:
    print("Wrong type refused:", exc)

try:
    mine.register(ToolSpec("word_count", "duplicate", {"type": "object"}, "read"), lambda: None)
except ValueError as exc:
    print("Duplicate registration refused:", exc)

### Checkpoint

**1. The registry validated the arguments. Why is policy still needed?**

<details><summary>Show answer</summary>

Validation only answers *is this well-formed?*. `send_email` with a valid address and a valid body is perfectly well-formed and may still be something this agent must not do unsupervised. Shape and permission are different questions, decided by different code.

</details>

**2. Why does `additionalProperties: False` matter?**

<details><summary>Show answer</summary>

Without it, a model can attach arguments nobody declared, and they flow straight into your function call. Declaring the schema closed means an unexpected argument is a rejected call rather than a surprise keyword argument.

</details>

### Recap

- Limitation: hardcoded if/elif tool dispatch cannot give two agents different tools, and has nowhere to record a tool's risk.
- Layer added: a registry holding name, description, schema, risk and function, with per-agent scoped discovery and structural validation.
- Evidence: four tools registered, one visible to the research agent, and four different malformed calls rejected before any function body ran.


---

### Section 5.3 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-4"></a>

## 5.4 — Permissions, Approval and Limits

The model proposes; Python disposes. That single sentence is the safety model of this
course, and this lesson is where it becomes running code.

We will watch an external action pause, a rejection end cleanly, a destructive tool be
refused even when it is allow-listed, an unrecognised risk level fail closed, and a
runaway loop get stopped by a limit the configuration cannot raise.


## Before you begin

### Learning outcomes

- Turn a tool's risk level into an allow / approval / deny decision.
- Pause a run on a checkpoint and resolve it in both directions.
- Show that limits and unknown inputs both fail closed.

Architecture reference: [Day 5 diagrams D16](../diagrams/source/day_05.md).

### Expected observation

`send_email` pauses at `pending_approval`; rejecting produces the status `cancelled`; `erase_workspace` is denied while allow-listed; a config asking for 500 steps is held to the runtime's cap.


## Concept briefing

## Permissions, approval and limits

The model proposes an action; Python decides whether it may happen. That decision
uses two independent facts: is the tool on this agent's allow-list, and what is
the tool's local **risk level**? The mini harness maps risk to a decision in one
place, `policy.RISK_POLICY`:

| Risk | Meaning | Decision |
|---|---|---|
| `read` | no effect outside the process | allow |
| `write` | reversible local change | allow |
| `external` | leaves the machine or is visible to others | approval |
| `destructive` | irreversible | deny |

Anything not in that table returns `deny`. That is **failing closed**: an unknown
or misspelled risk label must never be read as permission. A policy that raises
an exception on an unfamiliar input is worse, because a crash in the wrong place
can be caught and ignored, while an explicit `deny` is a decision that gets logged.

`approval` is not a question asked in the conversation. The run stops, the exact
pending action is written to a checkpoint, and a separate `resume` call carries
the human answer. Rejection is a **successful** safety outcome, so it has its own
status, `cancelled`, distinct from `failed`.

Limits are the other half of control. A configuration may request any number of
steps, but the runtime owns a hard ceiling (`MAX_STEPS_HARD_CAP`); the effective
limit is the smaller of the two, and that is the number the events report. A loop
that ends at `step_limit` has not crashed - it has been stopped on purpose.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — The whole policy, printed

`policy.decide` is under ten lines. Print the table it uses before trusting anything else.


In [ ]:
from mini_harness import RISK_POLICY, build_demo_registry, decide

print("risk level  -> decision")
for risk, decision in RISK_POLICY.items():
    print(f"  {risk:<12}-> {decision}")

print()
registry = build_demo_registry()
task = load_config("task_agent")
print("For task_agent, tool by tool:")
for spec in registry.discover():
    on_list = spec.name in task.allowed_tools
    print(f"  {spec.name:<16} risk={spec.risk:<12} allow-listed={str(on_list):<5} "
          f"decision={decide(task, spec)}")
print()
print("Note create_draft and send_email are both allow-listed, and get different answers.")

## Step 2 — An external action pauses the run

`send_email` is classified `external`. The run does not fail and does not ask the model
for permission. It stops, and writes down exactly what it wanted to do.


In [ ]:
from mini_harness import HarnessRuntime, MockModel

runtime = HarnessRuntime(build_demo_registry(), MockModel())
pending = runtime.run(task, "Send a synthetic course update")

print("Status        :", pending.status)
print("Paused tool   :", pending.pending_action["tool"])
print("Exact arguments the human is being asked to approve:")
for key, value in pending.pending_action["arguments"].items():
    print(f"    {key}: {value}")
print()
print("Saved to the checkpoint store under run id", pending.run_id)
print("Checkpoint keys:", sorted(runtime.checkpoints.load(pending.run_id)))

## Step 3 — Rejecting is a success, and it has its own status

Read the status carefully. A rejected run is **`cancelled`**, not `failed`. Nothing went
wrong; a person said no, and the harness records that as a distinct outcome.


In [ ]:
rejected = runtime.resume(pending.run_id, task, approved=False)

print("Status after rejection:", rejected.status)
print("Message               :", rejected.output)
print()
print("Is that the same as a failure?", rejected.status == "failed")
print("The five end states a run can reach are:")
print("  completed | pending_approval | cancelled | failed | step_limit")
print()
print("Last three events:")
for event in rejected.events[-3:]:
    print(f"  {event['event']:<20}",
          {k: v for k, v in event["details"].items() if k != "history"})
print()
print("Checkpoint after resolving:", runtime.checkpoints.load(pending.run_id))
print("-> cleared, so the same action cannot be approved twice.")

## Step 4 — Allow-listing a destructive tool changes nothing

A common misunderstanding is that the allow-list *is* the permission system. It is only
half of it: the tool's risk level is the other half, and `destructive` always loses.


In [ ]:
greedy = load_config("task_agent")
greedy.allowed_tools.append("erase_workspace")     # deliberately over-permissive config

destructive = registry.get("erase_workspace").spec
print("erase_workspace now on the allow-list:", "erase_workspace" in greedy.allowed_tools)
print("It is therefore discoverable        :",
      "erase_workspace" in [s.name for s in registry.discover(greedy.allowed_tools)])
print("Policy decision                     :", decide(greedy, destructive))
print()
# Now make the agent actually ask for it, so we see what the RUN does, not just
# what the policy function returns. (mock_plan is how we steer the mock model.)
greedy.mock_plan = [{"tool": "erase_workspace", "arguments": {}}]
result = runtime.run(greedy, "Erase everything")
print("A run that genuinely requests erase_workspace:")
print("  status:", result.status)
print("  reason:", result.output)
print("  events:", [e["event"] for e in result.events])
print()
print("The tool function never ran. 'deny' stops the run before execution,")
print("and the denial itself is recorded as a policy_decision event.")

## Step 5 — An unrecognised risk level fails closed

New tools arrive from places you do not control (Day 5.6 imports them over MCP). What if
one carries a risk label the policy table has never seen?


In [ ]:
from mini_harness import AgentConfig, ToolSpec

# A tool whose risk label is not one of read/write/external/destructive.
mystery = ToolSpec("mystery_action", "A capability from somewhere else",
                   {"type": "object", "properties": {}}, "quantum")
somebody = AgentConfig("somebody", "Try anything.", ["mystery_action"])

print("Is 'quantum' in the policy table?", "quantum" in RISK_POLICY)
print("Decision for an unknown risk    :", decide(somebody, mystery))
print()
print("It denies. It does not raise, and it certainly does not allow.")
print("That is 'failing closed': the safe answer for an input we do not understand.")
print("An exception here would be worse - exceptions get caught and swallowed,")
print("while a 'deny' is a decision that gets recorded in the event log.")

## Step 6 — Limits the configuration cannot raise

A loop that never terminates is not a hypothetical. Here is a model that always asks for
the same tool again, and two different ways the harness stops it.


In [ ]:
from mini_harness import MAX_STEPS_HARD_CAP, ModelDecision, effective_step_limit

class EndlessModel:
    """Always asks for the same tool. Never says 'done'."""
    def decide(self, prompt, config, tools, history):
        return ModelDecision("tool", tool="lookup_notes", arguments={"query": prompt})

print("Hard cap compiled into runtime.py:", MAX_STEPS_HARD_CAP)
print()

for requested in (2, 500):
    config = load_config("research_agent")
    config.max_steps = requested
    limit = effective_step_limit(config)
    print(f"config asks for {requested:>3} steps -> effective limit {limit}")

print()
looping = load_config("research_agent")
looping.max_steps = 2
outcome = HarnessRuntime(build_demo_registry(), EndlessModel()).run(looping, "keep going")
print("Status      :", outcome.status)
print("First event :", outcome.events[0]["event"], outcome.events[0]["details"])
print("Last event  :", outcome.events[-1]["event"], outcome.events[-1]["details"])
print()
print("The event reports the EFFECTIVE limit, so the log never disagrees with reality.")

### Try it yourself

Step 3 rejected the pending email. Predict what changes in the event trace if you approve
it instead — and what the final status becomes.


In [ ]:
# --- Worked solution ---
# Start a fresh run, because the earlier checkpoint was consumed by the rejection.
approved_run = runtime.run(task, "Send a synthetic course update")
print("Paused again at:", approved_run.status, "->", approved_run.pending_action["tool"])

# resume() carries the human answer. The model is NOT asked again: the exact
# arguments come back out of the checkpoint, not out of a new model call.
final = runtime.resume(approved_run.run_id, task, approved=True)

print()
print("Final status:", final.status)
print("Tool output :", final.output)
print()
print("Event trace, rejection vs approval:")
print("  rejected :", [e["event"] for e in rejected.events[-3:]])
print("  approved :", [e["event"] for e in final.events[-4:]])
print()
print("Both paths record approval_resolved. Only the approved one reaches tool_completed,")
print("and only then does the side effect happen.")

### Checkpoint

**1. Why must the approval decision come back through `resume()` rather than by asking the model again?**

<details><summary>Show answer</summary>

Because the model would be re-deciding, not confirming. The checkpoint holds the exact arguments a human reviewed. Re-prompting could produce a different recipient or body, and the human would have approved something that never ran.

</details>

**2. A tool arrives from an outside server carrying `risk="maybe"`. What happens?**

<details><summary>Show answer</summary>

`decide` returns `deny`, because `RISK_POLICY.get(risk, 'deny')` falls back to denial for anything it does not recognise. An unknown label can never be read as permission, and the denial is recorded as a normal policy decision rather than as a crash.

</details>

### Recap

- Limitation: an allow-list on its own cannot express 'visible, but not without a human', and a config could otherwise grant itself unlimited steps.
- Layer added: a risk-to-decision table that fails closed, a checkpointed approval pause, and a hard step cap owned by the runtime.
- Evidence: `send_email` paused and resolved to `cancelled` then `completed`; an allow-listed `erase_workspace` was still denied; an unknown risk denied; a 500-step request was held to 10.


---

### Section 5.4 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-5"></a>

## 5.5 — Events, Logs and Checkpoints

Two records that are easy to confuse. **Events** are append-only observations: they explain
what happened and are never edited. A **checkpoint** is mutable continuation state: it is
what a paused run needs in order to carry on later, and it is deleted once it is used.

This lesson also makes the runtime's retry budget visible, because "the model call failed"
is the most common thing that will ever happen to your agent.


## Before you begin

### Learning outcomes

- Write a durable JSONL event log and read it back.
- Resume a paused run after rebuilding the runtime from scratch.
- Watch a bounded retry with exponential backoff, and see it give up.

Architecture reference: [Day 5 diagrams D16](../diagrams/source/day_05.md).

### Expected observation

The event file survives on disk; a rebuilt runtime resumes the paused approval; a provider that fails once is retried and succeeds, and one that always fails stops after a fixed number of attempts.


## Concept briefing

## Events and checkpoints

Events are append-only observations such as run started, model completed, policy decided
and tool completed. A trace groups events belonging to one run. A checkpoint stores
continuation state so a paused run can resume.

A checkpoint is not an audit log, and an event log is not enough to resume execution.
Durable approval needs the exact pending action and a stable run identifier. Sensitive
arguments should be redacted or omitted from telemetry where possible.

## Retries, timeouts and retry budgets

Network calls fail. The runtime applies a timeout and retries transient failures
such as temporary rate limits, with **exponential backoff**: each attempt waits
twice as long as the previous one. Every attempt is recorded as a `provider_retry`
event, and the budget is bounded by `MAX_PROVIDER_RETRIES`; exhausting it emits
`provider_retry_budget_exhausted` and ends the run. In production, a small random
`jitter` is added to the delay so many clients do not all retry at the same instant.

Do not retry every failure. Invalid arguments, unknown tools and most configuration
or authentication errors will not improve on repetition, so the runtime records
`provider_error_not_retried` and stops at once. Consequential tools need an
idempotency strategy before any automatic retry. Step budget and retry budget are
separate limits so one failing provider cannot consume unlimited time or credit.

## Cost attribution

Record model, configuration version, input tokens, output tokens, reasoning tokens,
estimated cost and run ID. That makes it possible to compare agents and enforce
classroom budgets. Cost belongs to the complete run, including retries, not only
the final response - which is why usage is attached to every `model_completed`
event rather than to the result.

Mock mode reports empty usage on purpose: nothing was bought, so nothing is
counted. The included API credit is a controlled learning resource. Use mock mode
while debugging application logic; spend credit only when model behaviour itself
is the subject of the exercise.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — A fresh, disposable run folder

Every artefact this notebook writes goes under `data/generated/`, which the repository
ignores. A timestamped folder means re-running the notebook never collides with itself.


In [ ]:
from datetime import datetime

RUN_ROOT = PROJECT_ROOT / "data" / "generated" / ("events_lesson_" +
            datetime.now().strftime("%Y%m%d_%H%M%S"))
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("This run writes only inside:", RUN_ROOT)
print("Nothing outside data/generated/ is touched, so the lesson is safely repeatable.")

## Step 2 — Events go to a file as well as to memory

Give `EventStore` a path and every event is appended to a JSON-lines file the moment it
happens. Crash the process and the record of what happened is still on disk.


In [ ]:
from mini_harness import (EventStore, HarnessRuntime, JSONCheckpointStore,
                          MockModel, build_demo_registry, effective_step_limit)

events = EventStore(RUN_ROOT / "events.jsonl")
checkpoints = JSONCheckpointStore(RUN_ROOT / "checkpoints")
runtime = HarnessRuntime(build_demo_registry(), MockModel(), events, checkpoints)

task = load_config("task_agent")
print("Effective step limit for this run:", effective_step_limit(task))

paused = runtime.run(task, "Send the synthetic update")
print("Status:", paused.status)
print()
for event in events.get(paused.run_id):
    print(f"{event['event']:<20}", event["details"])

In [ ]:
# The same events are already durable on disk, one JSON object per line.
import json

lines = (RUN_ROOT / "events.jsonl").read_text(encoding="utf-8").splitlines()
print("Lines written to events.jsonl:", len(lines))
print()
print("First line, raw:")
print(" ", lines[0])
print()
print("Parsed back into Python:")
first = json.loads(lines[0])
print("  run_id   :", first["run_id"])
print("  event    :", first["event"])
print("  timestamp:", first["timestamp"])

## Step 3 — The checkpoint is what makes resuming possible

An event log tells you a run paused. It is not enough to *continue* it. The checkpoint
holds the exact pending action.


In [ ]:
state = checkpoints.load(paused.run_id)
print("Checkpoint contents:")
for key in sorted(state):
    value = state[key]
    shown = f"<{len(value)} history messages>" if key == "history" else value
    print(f"  {key:<12}:", shown)

print()
print("On disk at:", RUN_ROOT / "checkpoints" / f"{paused.run_id}.json")

In [ ]:
# Simulate a restart: throw the runtime away and build a brand-new one that shares
# only the folder on disk. The pending approval must survive that.
restarted = HarnessRuntime(build_demo_registry(), MockModel(),
                           events, JSONCheckpointStore(RUN_ROOT / "checkpoints"))

print("New runtime object:", id(restarted) != id(runtime))
done = restarted.resume(paused.run_id, task, approved=True)
print("Resumed status    :", done.status)
print("Tool output       :", done.output)
print("Checkpoint after  :", checkpoints.load(paused.run_id), "(consumed, so it cannot replay)")
print()
print("Full event trace across BOTH runtime objects:")
print(" ", [e["event"] for e in events.get(paused.run_id)])

## Step 4 — A transient failure is retried, with backoff

`FlakyModel` is a teaching double: it raises on its first call and then behaves like the
normal mock. Watch the runtime absorb that without the run failing.


In [ ]:
from mini_harness import FlakyModel, MAX_PROVIDER_RETRIES

print("Retry budget per model call:", MAX_PROVIDER_RETRIES, "extra attempts")

retry_events = EventStore()
flaky = HarnessRuntime(build_demo_registry(), FlakyModel(failures=1), retry_events)
recovered = flaky.run(load_config("research_agent"), "What is a harness?")

print("Final status:", recovered.status)
print()
for event in recovered.events:
    if event["event"].startswith("provider") or event["event"] in {"model_completed", "run_completed"}:
        print(f"{event['event']:<28}", event["details"])
print()
print("Note retry_in_seconds doubles each attempt. That is exponential backoff:")
print("  attempt 1 waits 0.05s, attempt 2 waits 0.10s, attempt 3 waits 0.20s ...")
print("Production code also adds a small random 'jitter' so that many clients")
print("recovering from the same outage do not all retry at the same instant.")

## Step 5 — The budget is bounded, and some errors are not retried at all

Retrying forever is just a slower outage. And retrying the *wrong* errors wastes time and
credit on something that cannot improve.


In [ ]:
always_failing = HarnessRuntime(build_demo_registry(), FlakyModel(failures=99), EventStore())
gave_up = always_failing.run(load_config("research_agent"), "What is a harness?")

print("Status:", gave_up.status)
print("Attempts recorded:", len([e for e in gave_up.events if e["event"] == "provider_retry"]) + 1)
for event in gave_up.events:
    if event["event"].startswith(("provider", "run_failed")):
        print(f"  {event['event']:<34}", event["details"].get("error", ""))

print()
# A ValueError means bad arguments or bad configuration. Repeating it changes nothing.
class MisconfiguredModel:
    def decide(self, prompt, config, tools, history):
        raise ValueError("model name is not valid for this provider")

not_retried = HarnessRuntime(build_demo_registry(), MisconfiguredModel(), EventStore())
result = not_retried.run(load_config("research_agent"), "hello")
print("Non-transient error status:", result.status)
print("Events:", [e["event"] for e in result.events])
print("-> provider_error_not_retried: the harness stops immediately instead of burning")
print("   the retry budget on an error that will never succeed.")

## Step 6 — Where cost lives

Cost belongs to a whole run — including the retries you just watched — not to the final
reply. That is why usage is attached to every `model_completed` event.


In [ ]:
totals = {"prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0}
for event in recovered.events:
    if event["event"] == "model_completed":
        for key in totals:
            totals[key] += event["details"]["usage"].get(key, 0)

print("Model calls in that run:",
      len([e for e in recovered.events if e["event"] == "model_completed"]))
print("Totals attributed to run", recovered.run_id[:8], ":", totals)
print()
print("Zero, because this was MOCK mode - nothing was bought. In LIVE mode the same")
print("three lines give you a per-run bill, and because failed attempts also emit")
print("events you can see what an outage actually cost you.")

### Try it yourself

Predict: if you delete the checkpoint file before resuming, what does `resume()` do?


In [ ]:
# --- Worked solution ---
# Pause a fresh run, delete its checkpoint, then try to resume it.
victim = runtime.run(task, "Send another synthetic update")
path = RUN_ROOT / "checkpoints" / f"{victim.run_id}.json"
print("Paused          :", victim.status)
print("Checkpoint file :", path.name, "exists:", path.exists())

path.unlink()                       # simulate losing the durable state
print("Deleted it. exists:", path.exists())

outcome = runtime.resume(victim.run_id, task, approved=True)
print()
print("Status:", outcome.status)
print("Reason:", outcome.output)
print()
print("It fails closed. Without the exact pending action there is nothing safe to run,")
print("and the harness will not ask the model to invent it again.")
print("The EVENT log still shows the run paused - events explain, checkpoints continue.")

### Checkpoint

**1. You have the full event log for a paused run. Can you resume it?**

<details><summary>Show answer</summary>

No. The event log explains what happened; it is an audit record, and it deliberately does not promise to hold everything needed to continue. Resuming needs the checkpoint: the exact tool, the exact arguments and the conversation history, stored under a stable run id.

</details>

**2. Why does the runtime retry a `RuntimeError` but not a `ValueError`?**

<details><summary>Show answer</summary>

A `RuntimeError` from the provider means the network call itself failed - a timeout or a rate limit - and the same request may well succeed a moment later. A `ValueError` means the request was malformed or misconfigured; sending it again produces the identical error, so the harness records `provider_error_not_retried` and stops.

</details>

### Recap

- Limitation: printed output disappears with the kernel, and a paused run had nothing durable to continue from.
- Layer added: a JSONL event log, a JSON checkpoint store, and a bounded retry budget with exponential backoff, all recorded as events.
- Evidence: a brand-new runtime resumed a paused approval from disk; a single transient failure was retried and recovered; a permanently failing provider stopped after a fixed number of attempts.


---

### Section 5.5 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-6"></a>

## 5.6 — MCP Client: Discover, Then Govern

Model Context Protocol is a standard way for a client to ask a server *what can you do?*
and then *do this*. It is genuinely useful: one client can talk to tools nobody on your
team wrote.

It also grants exactly no authority. The server describes itself; your harness decides
what that description is worth.


## Before you begin

### Learning outcomes

- Discover a tool over an MCP-shaped client and read its schema.
- Classify a discovered tool locally and run it past your own policy.
- Connect to a real stdio MCP server, or degrade cleanly when the SDK is absent.

Architecture reference: [Day 5 diagrams D17](../diagrams/source/day_05.md).

### Expected observation

`course_lookup` is discovered, classified `read`, allowed, and called. Re-classifying the same tool as `external` changes the decision while discovery is unchanged.


## Concept briefing

## MCP: protocol, not permission

Model Context Protocol lets a client initialise a session, discover server capabilities
and invoke them through a common contract. A server may expose tools, resources or prompts.
The protocol improves interoperability; it does not establish trust.

An MCP tool description and its results are untrusted external content. Before importing
a discovered tool, the harness should consider server origin, schema, local risk,
permitted agents, arguments, timeout, output handling and logging. A server changing its
advertised tools must not silently expand application authority.

The Day 5 rule is therefore:

```text
discovery is not authorization
```

The client discovers the tool, the harness classifies it, local policy authorises or
pauses it, and only then does the protocol call occur.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — Discovery, offline

`FakeMCPClient` speaks the same shape as the real SDK: a list of tool objects with
`.name`, `.description` and `.inputSchema`. Learning one access pattern is the point.


In [ ]:
from mini_harness import FakeMCPClient

client = FakeMCPClient()
tools = await client.list_tools()          # notebooks allow top-level await

print("Tools advertised by the server:", len(tools))
for tool in tools:
    print("  object type :", type(tool).__name__)
    print("  .name       :", tool.name)
    print("  .description:", tool.description)
    print("  .inputSchema:", tool.inputSchema)

print()
print("Everything above is text the SERVER chose to send us. It is evidence, not truth.")

## Step 2 — Classify it yourself before it enters the harness

The server does not get to say how risky its own tool is. We convert the description into
a local `ToolSpec` and assign the risk level ourselves.


In [ ]:
from mini_harness import AgentConfig, ToolSpec, decide

remote = tools[0]
spec = ToolSpec(
    name=remote.name,
    description=remote.description,
    input_schema=remote.inputSchema,
    risk="read",                    # OUR judgement: it only returns a fact
)
mcp_agent = AgentConfig("mcp_demo", "Use one supplied fact.", allowed_tools=[spec.name])

print("Imported as   :", spec)
print("Local risk    :", spec.risk, "(assigned by us, not by the server)")
print("Policy decision:", decide(mcp_agent, spec))

## Step 3 — Only now do we call it

Note the order: discover, classify, authorise, *then* invoke. `tool_result_payload`
normalises the answer so the fake and a real server are read the same way.


In [ ]:
from mini_harness import tool_result_payload

if decide(mcp_agent, spec) == "allow":
    raw = await client.call_tool(spec.name, {"topic": "mcp"})
    print("Raw result object:", type(raw).__name__)
    print("Normalised payload:", tool_result_payload(raw))
else:
    print("Policy refused; no protocol call was made.")

## Step 4 — Change the classification, not the server

The server is untouched. The description is identical. Only our local risk label moved,
and the decision moved with it.


In [ ]:
reclassified = ToolSpec(remote.name, remote.description, remote.inputSchema, "external")

print("Discovery is unchanged:", reclassified.name == spec.name,
      "| same schema:", reclassified.input_schema == spec.input_schema)
print()
print("risk = read     -> decision:", decide(mcp_agent, spec))
print("risk = external -> decision:", decide(mcp_agent, reclassified))
print()
print("This is 'discovery is not authorization' in one comparison.")
print("A server that renames a tool, or quietly changes what it does, cannot")
print("expand what your agent is permitted to do - because it never set the risk.")

## Step 5 — A real MCP server over stdio

`instructor_mcp_server.py` is a genuine MCP server. This cell launches it as a subprocess,
completes a protocol session and calls one tool. Two details that matter on Windows: the
command must be `sys.executable` (the interpreter running this notebook, never the literal
string `"python"`), and the session runs in `..._sync`, which gives the subprocess its own
event loop. If the SDK is not installed, the cell says how to install it and moves on.


In [ ]:
import importlib.util

MCP_AVAILABLE = importlib.util.find_spec("mcp") is not None
print("MCP SDK installed:", MCP_AVAILABLE)

if not MCP_AVAILABLE:
    print('Optional: pip install "mcp>=1.27,<2" to run this cell against a real server.')
    print("Everything above already ran against FakeMCPClient, so nothing is missing.")
else:
    from mini_harness import StdioMCPClient
    server = PROJECT_ROOT / "instructor_mcp_server.py"
    real = StdioMCPClient(sys.executable, [str(server)])   # sys.executable, not "python"
    try:
        real_tools, real_result = real.list_and_optionally_call_sync(
            "course_lookup", {"topic": "harness"})
        print("Server:", server.name)
        for tool in real_tools:
            print("  object type :", type(tool).__name__, "(from the real SDK)")
            print("  .name       :", tool.name)
            print("  .description:", tool.description)
            print("  .inputSchema:", tool.inputSchema)
        print("Result payload:", tool_result_payload(real_result))
        print()
        print("Same three attributes as the fake, read with the same code.")
        print("That is why Step 1 was worth doing offline first.")
    except Exception as exc:                 # noqa: BLE001 - never stop the class
        print("Real MCP session failed:", type(exc).__name__, exc)
        print("Use the FakeMCPClient path above; the lesson is unchanged.")

Note what did **not** change. The real server returned real objects, and the
policy step in Step 2 would be identical: classify it locally, then ask `decide`.
Nothing about "this came from a real server" makes it more trusted.


### Try it yourself

Predict what happens when you ask an MCP server for a tool it never advertised.


In [ ]:
# --- Worked solution ---
# Two separate protections, and they fail at different moments.

# 1. Asking the client directly for an unknown tool: the server refuses.
try:
    await client.call_tool("delete_everything", {})
except KeyError as exc:
    print("Server side  : unknown tool ->", type(exc).__name__, exc)

# 2. The more important case: the tool IS advertised, but is not in our allow-list.
sneaky = ToolSpec("delete_everything", "Advertised by the server", {"type": "object"}, "read")
print("Advertised by the server, but on our allow-list?",
      sneaky.name in mcp_agent.allowed_tools)
print("Local policy decision:", decide(mcp_agent, sneaky))
print()
print("Even labelled 'read', it is denied - because policy checks the allow-list first.")
print("A server can advertise anything it likes; it cannot add itself to your allow-list.")

### Checkpoint

**1. An MCP server updates and now advertises a `publish_site` tool. What can it do?**

<details><summary>Show answer</summary>

Nothing, on its own. Discovery only tells your harness the tool exists. It is not in any agent's `allowed_tools`, and nobody has given it a local risk level, so `decide` returns `deny`. Someone has to make a deliberate change to your configuration before it can run.

</details>

**2. Why does the fake client return objects rather than plain dictionaries?**

<details><summary>Show answer</summary>

Because the real SDK returns objects with `.name`, `.description` and `.inputSchema`. If the fake returned dictionaries you would learn `tool["name"]`, and every line of that code would break the first time you pointed it at a real server.

</details>

### Recap

- Limitation: a protocol makes outside capabilities reachable, which is exactly what makes it dangerous - the server writes its own description.
- Layer added: local classification of every discovered tool into a `ToolSpec`, checked by the same `policy.decide` used for local tools.
- Evidence: one tool discovered and called only after being allowed; re-labelling it `external` changed the decision without touching the server; an advertised but un-allow-listed tool was denied.


---

### Section 5.6 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-7"></a>

## 5.7 — Day 5 Project — Mini AI Harness

Everything from 5.1 to 5.6 in one place. One runtime hosts two agent configurations, a
registry supplies scoped tools, policy governs them, events explain the runs, a checkpoint
carries an approval, and MCP sits behind the same boundary as everything else.

Then you add a third agent — and change no runtime code at all.


## Before you begin

### Learning outcomes

- Run two different agents through a single runtime, registry and policy.
- Follow one approval from pause to checkpoint to resolution.
- Add a third agent configuration without editing a single line of the harness.

Architecture reference: [Day 5 diagrams D16-D18](../diagrams/source/day_05.md).

### Expected observation

The research agent completes with tool evidence; the task agent pauses on an external action; a newly written third configuration produces its own tool requests and policy events.


## Concept briefing

## Mapping the course to production systems

| Course term | Common production terminology |
|---|---|
| Provider adapter | model client/provider layer |
| Agent configuration | agent definition/profile |
| Harness runtime | agent runtime/orchestration layer |
| Tool registry | tool/plugin registry |
| Policy | authorization or guardrail middleware |
| Events | tracing/telemetry |
| Checkpoint store | durable execution/state persistence |
| MCP client | protocol integration layer |

Production SDKs package different subsets of these responsibilities. Students should be
able to open an unfamiliar SDK and locate where its model calls, tools, policy, state and
events live rather than assuming the SDK itself is the architecture.

## What the mini harness does not provide

The classroom harness is intentionally not a production platform. It does not provide
enterprise identity, operating-system sandboxing, remote MCP authentication, distributed
workers, deployment or guaranteed model quality. Its purpose is to make the essential
boundaries visible so students can recognise and evaluate larger systems later.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — One runtime, two agents

Build the harness once. Everything after this reuses these three objects.


In [ ]:
from mini_harness import EventStore, HarnessRuntime, MockModel, build_demo_registry

registry = build_demo_registry()
events = EventStore()
runtime = HarnessRuntime(registry, MockModel(), events)

print("Registry holds :", [s.name for s in registry.discover()])
print("Runtime object :", type(runtime).__name__)
print()

for name, prompt in [("research_agent", "What is a harness?"),
                     ("task_agent", "Prepare a concise project update")]:
    config = load_config(name)
    result = runtime.run(config, prompt)
    print(f"--- {name} ---")
    print("  visible tools:", [s.name for s in registry.discover(config.allowed_tools)])
    print("  status       :", result.status)
    print("  output       :", result.output)
    print("  events       :", [e["event"] for e in result.events])
    print()

## Step 2 — The approval path, end to end

The external action pauses, the checkpoint holds the exact arguments, and a separate call
carries the human decision.


In [ ]:
task = load_config("task_agent")
pending = runtime.run(task, "Send a synthetic project update")

print("Status        :", pending.status)
print("Approval card :")
print("  tool     :", pending.pending_action["tool"])
print("  arguments:", pending.pending_action["arguments"])
print("  requested by agent:", pending.pending_action["agent"], "at step",
      pending.pending_action["steps_used"])

final = runtime.resume(pending.run_id, task, approved=True)
print()
print("After approval:", final.status)
print("Tool output   :", final.output)
print("Trace         :", [e["event"] for e in final.events])

## Step 3 — The two small pieces we have not used yet

A memory interface and an MCP boundary. Both are deliberately minimal: their job is to
show *where* the responsibility sits, not to be good at it.


In [ ]:
from mini_harness import FakeMCPClient, SimpleMemory, tool_result_payload

memory = SimpleMemory()
memory.add("fictional_asha", "Prefer concise project updates")
memory.add("fictional_asha", "Lab reports are due on Fridays")
print("Memory recall for 'concise update':", memory.search("fictional_asha", "concise update"))

client = FakeMCPClient()
discovered = await client.list_tools()
print("MCP discovery:", [t.name for t in discovered])
print("MCP call     :", tool_result_payload(await client.call_tool("course_lookup",
                                                                  {"topic": "harness"})))
print()
print("Both sit OUTSIDE the loop. Neither can bypass policy to cause a side effect.")

## Step 4 — What the whole day looks like as one trace

Every event recorded by the shared `EventStore`, grouped by run.


In [ ]:
print(f"Runs recorded in this EventStore: {len(events.by_run)}")
print()
for run_id, rows in events.by_run.items():
    agent = rows[0]["details"].get("agent", "?")
    print(f"run {run_id[:8]}  agent={agent}")
    for row in rows:
        detail = row["details"]
        note = detail.get("decision") or detail.get("tool") or detail.get("error") or ""
        print(f"    {row['event']:<20} {note}")
    print()

### Try it yourself

Add a **third** agent configuration — one that looks something up and then wants to email
the result — without editing `runtime.py`, `policy.py` or `providers.py`.


In [ ]:
# --- Worked solution ---
# A configuration is DATA, so a third agent is a new JSON document. Nothing in
# src/mini_harness/ changes. The `mock_plan` field is what lets the deterministic
# mock model act on a config it has never seen before.
import json

notes_agent_json = json.dumps({
    "name": "notes_agent",
    "instructions": "Look up a course note, then request that it be emailed.",
    "allowed_tools": ["lookup_notes", "send_email"],   # a read tool and an external one
    "max_steps": 4,
    "model": {"provider": "mock", "model": "mock-deterministic",
              "temperature": 0.0, "max_output_tokens": 300},
    "mock_plan": [
        # Step 1: a read tool -> policy will allow it outright.
        {"tool": "lookup_notes", "arguments": {"query": "{prompt}"}},
        # Step 2: an external tool -> policy will pause for approval.
        {"tool": "send_email", "arguments": {"to": "mentor@example.test",
                                             "subject": "Course note",
                                             "body": "{prompt}"}},
    ],
}, indent=2)
print(notes_agent_json)

In [ ]:
# --- Worked solution, part 2: run it ---
# Save it beside the other configurations so load_config() can find it, then run it
# through the SAME runtime object used in Step 1.
from mini_harness import AgentConfig, ModelConfig

raw = json.loads(notes_agent_json)
raw["model"] = ModelConfig(**raw["model"])
notes_agent = AgentConfig(**raw)

result = runtime.run(notes_agent, "harness")

print("Status:", result.status)
print()
print("Policy decisions this new agent produced:")
for event in result.events:
    if event["event"] == "policy_decision":
        d = event["details"]
        print(f"  {d['tool']:<16} risk={d['risk']:<10} -> {d['decision']}")
print()
print("Paused on:", result.pending_action["tool"])
print("Full trace:", [e["event"] for e in result.events])
print()
print("Lines of harness code changed to support a brand-new agent: 0")
print("(To keep it, write notes_agent_json to configs/notes_agent.json and use")
print(" load_config('notes_agent') from then on.)")

### Checkpoint

**1. A colleague says "just add the new agent logic to runtime.py". What is wrong with that?**

<details><summary>Show answer</summary>

The runtime would start containing application-specific behaviour, so every new agent would mean editing shared, safety-critical code. The third agent above needed a JSON document and nothing else - which is exactly why the runtime stayed trustworthy.

</details>

**2. Which responsibility deliberately stays application-specific?**

<details><summary>Show answer</summary>

Which tools an agent may use, and what its instructions are. The harness owns the *mechanism* - discovery, validation, the risk-to-decision table, limits, events. It never decides that this particular agent should be allowed to send email.

</details>

### Recap

- Limitation: reusable infrastructure is only reusable if adding an application needs no change to the infrastructure.
- Layer added: the complete harness - config, provider, registry, policy, runtime, events, checkpoints, memory and an MCP boundary.
- Evidence: three different agents ran through one runtime object, one of them written during the lesson, and the approval pause behaved identically for all of them.


---

### Section 5.7 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-8"></a>

## 5.8 — Pivotal Exercise: Build a Capability-Aware Tool Registry

This is the day's one hands-on implementation lab. It uses no API key. Try the starter cell first; a fully commented reference solution follows the check so you can compare or catch up.


## Why this mechanism matters

A harness needs one controlled place for tool discovery and dispatch. The registry connects model-visible schemas to host-owned handlers while policy limits which capabilities a configuration receives.

## Contract

Reject duplicate registrations with `ValueError`. Show schemas only for granted capabilities. Raise `KeyError` for an unknown tool and `PermissionError` for an ungranted one before the handler is called.

Before coding, write one sentence predicting the easiest mistake to make.

In [ ]:
class ToolRegistry:
    def __init__(self):
        self._tools = {}   # name -> {"schema": ..., "handler": ..., "capability": ...}

    def register(self, name, schema, handler, capability):
        """Register once; a second registration of the same name raises ValueError."""
        raise NotImplementedError("Complete registration")

    def schemas_for(self, granted_capabilities):
        """Return the schemas of tools whose capability is granted (least-privilege discovery)."""
        raise NotImplementedError("Complete filtered discovery")

    def dispatch(self, name, arguments, granted_capabilities):
        """Unknown name -> KeyError. Ungranted capability -> PermissionError. Otherwise call the handler."""
        raise NotImplementedError("Complete protected dispatch")

## Behavioural check

Run this after completing the starter cell. If you have not finished, it prints a hint instead of failing. A passing check proves the listed contract examples, not every possible input.

In [ ]:
def run_checks():
    registry = ToolRegistry()
    registry.register("add", {"name": "add", "parameters": {"a": "number", "b": "number"}},
                      lambda a, b: a + b, "math.read")
    print("visible with math.read :", [s["name"] for s in registry.schemas_for({"math.read"})])
    print("visible with nothing   :", [s["name"] for s in registry.schemas_for(set())])
    assert len(registry.schemas_for({"math.read"})) == 1
    assert registry.schemas_for(set()) == []
    assert registry.dispatch("add", {"a": 4, "b": 5}, {"math.read"}) == 9

    for label, call, expected in [
        ("ungranted dispatch", lambda: registry.dispatch("add", {"a": 1, "b": 1}, set()), PermissionError),
        ("unknown tool", lambda: registry.dispatch("nope", {}, {"math.read"}), KeyError),
        ("duplicate registration", lambda: registry.register("add", {}, lambda: None, "math.read"), ValueError),
    ]:
        try:
            call()
        except expected as exc:
            print(f"{label:<23} -> {type(exc).__name__}: {exc}")
        else:
            raise AssertionError(f"{label} must raise {expected.__name__}")
    print("PASS: registry centralizes discovery, dispatch, and capability checks")

try:
    run_checks()
except NotImplementedError:
    print("Not implemented yet. Complete the starter cell above, or study the reference solution below and re-run this cell.")

## Reference solution

Read this even if your check passed: compare each commented line with your version, then re-run the check cell above.

In [ ]:
# --- Reference solution: read it line by line, then re-run the check cell above ---
class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, name, schema, handler, capability):
        if name in self._tools:                                    # one name, one handler
            raise ValueError(f"tool {name!r} is already registered")
        self._tools[name] = {"schema": schema, "handler": handler, "capability": capability}

    def schemas_for(self, granted_capabilities):
        # Discovery is filtered so the model is never tempted by tools it may not use ...
        return [t["schema"] for t in self._tools.values() if t["capability"] in granted_capabilities]

    def dispatch(self, name, arguments, granted_capabilities):
        if name not in self._tools:                                # fail closed on unknown names
            raise KeyError(f"unknown tool {name!r}")
        tool = self._tools[name]
        if tool["capability"] not in granted_capabilities:         # ... and enforced AGAIN here
            raise PermissionError(f"capability {tool['capability']!r} not granted for {name!r}")
        return tool["handler"](**arguments)                        # only now does the host run it

print("Reference ToolRegistry defined. Re-run the check cell above to see PASS.")

## Explain

**Why must filtered schemas and protected dispatch both exist?**

<details><summary>Show answer</summary>

Filtering schemas reduces temptation: the model never sees a tool it may not use. But a model can still name a hidden tool, and code paths other than the model can call dispatch. Only the check inside dispatch actually prevents execution.

</details>

**Why raise an exception instead of returning None for an ungranted call?**

<details><summary>Show answer</summary>

A silent None can be mistaken for a successful empty result. An exception stops the run, is recorded as an event, and forces the caller to handle the denial explicitly.

</details>

---

### Section 5.8 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-5-section-9"></a>

## 5.9 — Day 5 Capstone — Website Maintenance Agent

A scheduler runs this once a day. It checks a source of updates, and if something new
appeared it drafts a change to a website and asks a human to approve it.

The scheduling part is ordinary automation. The interesting part is everything you built
this week: the two actions are **registered harness tools** with honest risk levels, so
drafting is allowed automatically and publishing is not.


## Before you begin

### Learning outcomes

- Register a real workflow's actions as harness tools and drive them from the runtime.
- Read the events that prove policy paused the run before anything was published.
- Watch change detection process a backlog and then report no change.

Architecture reference: [Day 5 diagrams D19](../diagrams/source/day_05.md).

### Expected observation

`propose_update` is allowed and `publish_update` pauses; the website file does not exist until an explicit approval; a poisoned source is refused by a guardrail and the refusal appears in the event log.


## Concept briefing

## Automation is a trigger, not intelligence

A scheduler can start a run every day, but scheduling alone is ordinary automation. The
agentic decision is whether new evidence warrants a change and which permitted action to
propose. Policy then decides whether the exact proposal may proceed.

The Website Maintenance Agent demonstrates a production-shaped cycle at classroom scale:
fetch a real or cached public update, compare it with durable processed-item state, create
a structured website proposal, apply guardrails, pause for approval, write a real local
file, verify the result and record events. The scheduler should call one bounded `check`
operation; it should not contain hidden business logic.

An optional LLM judge may score whether the proposed update is faithful to its source.
That judge belongs after deterministic checks and before approval or publication. It is
advisory because it can be inconsistent, biased toward fluent text or influenced by the
content it evaluates. File-path, schema, source, build and permission checks remain
authoritative application code.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — A fresh folder for this run

Everything is written under `data/generated/`, in a folder named for this moment. Re-run
the notebook as often as you like: each run starts from an empty website and empty state.


In [ ]:
from datetime import datetime
from uuid import uuid4

RUN_ROOT = (PROJECT_ROOT / "data" / "generated" /
            f"website_{datetime.now():%Y%m%d_%H%M%S}_{uuid4().hex[:6]}")
SITE_ROOT = RUN_ROOT / "site"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("Run folder   :", RUN_ROOT.name)
print("Website root :", SITE_ROOT)
print("Fresh start  : site folder exists?", SITE_ROOT.exists())

## Step 2 — The source, and what "new" means

The cached source is a small JSON file so the lesson is repeatable. Durable state records
which items have already been handled; anything not in that list is new work.


In [ ]:
from mini_harness import CachedJSONSource, JSONStateStore

source = CachedJSONSource(PROJECT_ROOT / "data" / "website_updates.json")
state = JSONStateStore(RUN_ROOT / "state.json")

items = source.fetch()
print("Items available from the source:", len(items))
for item in items:
    print(f"  {item.item_id:<20} {item.title}")
    print(f"  {'':<20} from {item.url}")

current = state.load()
print()
print("Already processed:", current["processed_ids"] or "nothing yet (fresh state)")
unseen = [i for i in items if i.item_id not in current["processed_ids"]]
print("Therefore new    :", [i.item_id for i in unseen])

## Step 3 — Guardrails are the domain expert

Policy answers "may this *kind* of action happen?". Guardrails answer the questions only
this application can: is that host trusted, does the text contain instructions aimed at
the model, does the target path stay inside the website folder?


In [ ]:
from mini_harness import WebsiteGuardrails

guardrails = WebsiteGuardrails(SITE_ROOT, trusted_hosts={"github.com"})

print("Website root guarded  :", guardrails.site_root.name)
print("Trusted source hosts  :", guardrails.trusted_hosts)
print("Maximum body length   :", guardrails.max_body_chars, "characters")
print()
clean = unseen[0]
print("Input guardrail on", clean.item_id, "->", guardrails.check_source(clean) or "no failures")

## Step 4 — The two actions, registered as harness tools

This is the step that connects the capstone to the rest of Day 5. Drafting is reversible
and local, so it is `write`. Publishing changes the website, so it is `external`.


In [ ]:
from mini_harness import build_website_registry, deterministic_proposer, website_agent_config

# deterministic_proposer is honest about itself: it copies the cleaned-up source
# summary verbatim. There is NO model call on this path - which is exactly why the
# whole capstone runs with no API key. Step 8 swaps in the live proposer.
proposer = deterministic_proposer

registry = build_website_registry(source, proposer, guardrails, state)
config = website_agent_config()

print("Registered website tools:")
for spec in registry.discover():
    print(f"  {spec.name:<16} risk={spec.risk:<10} {spec.description}")
print()
print("Agent configuration:", config.name)
print("  allowed tools:", config.allowed_tools)
print("  plan          :", [step["tool"] for step in config.mock_plan])

In [ ]:
from mini_harness import decide

print("What policy will say before anything runs:")
for spec in registry.discover():
    print(f"  {spec.name:<16} -> {decide(config, spec)}")
print()
print("Nothing here is a special case. It is the same policy.decide from Day 5.4,")
print("reading the same risk levels, for a workflow that touches a real file.")

## Step 5 — One scheduled tick

The runtime drives both tools. Watch where it stops.


In [ ]:
from mini_harness import EventStore, HarnessRuntime, JSONCheckpointStore, MockModel

events = EventStore(RUN_ROOT / "events.jsonl")
runtime = HarnessRuntime(registry, MockModel(), events,
                         JSONCheckpointStore(RUN_ROOT / "checkpoints"))

target = SITE_ROOT / "content" / "updates.md"
print("Website file before the run:", "exists" if target.exists() else "does not exist")

result = runtime.run(config, clean.item_id)
print("Run status:", result.status)
print()
for event in result.events:
    d = event["details"]
    note = d.get("decision") or d.get("tool") or ""
    print(f"  {event['event']:<20} {note}")

In [ ]:
# The events are the evidence. Read them as a sequence of decisions.
print("Policy decisions, in order:")
for event in result.events:
    if event["event"] == "policy_decision":
        d = event["details"]
        print(f"  {d['tool']:<16} risk={d['risk']:<10} -> {d['decision']}")

print()
print("Paused on           :", result.pending_action["tool"])
print("Arguments held      :", result.pending_action["arguments"])
print("Website file exists :", target.exists(), "  <- observe: still nothing written")
print()
print("A draft WAS created, though - it is sitting in durable state, not on the site:")
print("  pending item ids:", list(state.load()["pending"]))

## Step 6 — Reject, then approve

Rejection first, so you can see that saying no really does leave the website alone.


In [ ]:
rejected = runtime.resume(result.run_id, config, approved=False)
print("Rejected run status :", rejected.status)
print("Website file exists :", target.exists())
print("Events at the end   :", [e["event"] for e in rejected.events[-3:]])

In [ ]:
# Now run the tick again and approve it. This is the only cell in the notebook
# that changes a file on disk, and it does so only because we said approved=True.
second = runtime.run(config, clean.item_id)
print("Paused again on:", second.pending_action["tool"])

approved = runtime.resume(second.run_id, config, approved=True)
print("Status after approval:", approved.status)
print("Tool result          :", approved.output)
print()
print("Website file exists  :", target.exists())
print("--- content/updates.md ---")
print(target.read_text(encoding="utf-8"))

## Step 7 — Change detection over several ticks

The cached source holds two items. A scheduler running daily should work through the
backlog and then go quiet.


In [ ]:
def one_tick(label):
    """Exactly what a scheduler does: find new work, or report that there is none."""
    processed = set(state.load()["processed_ids"])
    todo = [i for i in source.fetch() if i.item_id not in processed]
    print(f"{label:<10} processed={sorted(processed)}")
    if not todo:
        print(f"{'':<10} -> no_change (nothing new; no model call, no run)")
        return None
    run = runtime.run(config, todo[0].item_id)
    print(f"{'':<10} -> new item {todo[0].item_id}: {run.status} on {run.pending_action['tool']}")
    return run

tick2 = one_tick("tick 2")
runtime.resume(tick2.run_id, config, approved=True)
tick3 = one_tick("tick 3")

print()
print("--- final website ---")
print(target.read_text(encoding="utf-8"))

## Step 8 — A poisoned source is refused before anything is drafted

External text is evidence, never instructions. This fixture contains a real-looking
update with "ignore all previous instructions" buried inside it.


In [ ]:
poisoned_source = CachedJSONSource(PROJECT_ROOT / "data" / "poisoned_website_updates.json")
poisoned_registry = build_website_registry(
    poisoned_source, proposer, guardrails, JSONStateStore(RUN_ROOT / "poisoned_state.json"))
poisoned_runtime = HarnessRuntime(poisoned_registry, MockModel(), events)

bad_item = poisoned_source.fetch()[0]
print("Poisoned summary:", bad_item.summary[:90], "...")
print()

blocked = poisoned_runtime.run(website_agent_config(), bad_item.item_id)
print("Run status:", blocked.status)
print("Reason    :", blocked.output)
print()
for event in blocked.events:
    if event["event"] in {"policy_decision", "tool_failed", "run_failed"}:
        print(f"  {event['event']:<16}", event["details"])
print()
print("It never reached publish_update, because propose_update refused to draft at all.")
print("Website file still exists?", target.exists(), "(from Step 6/7, unchanged by this)")

## Step 9 — The live paths, both optional

Two independent live options: a real source (public GitHub releases) and a real model
(OpenRouter writing the update text). Both are guarded, both fall back, and neither can
publish anything — the run still stops at `pending_approval`.


In [ ]:
# Live SOURCE. Off by default: set RUN_LIVE_FETCH=1 in your environment to try it.
from mini_harness import GitHubReleaseSource

RUN_LIVE_FETCH = os.getenv("RUN_LIVE_FETCH") == "1"
print("RUN_LIVE_FETCH =", RUN_LIVE_FETCH)

live_items = []
if RUN_LIVE_FETCH:
    try:
        live_items = GitHubReleaseSource("modelcontextprotocol", "python-sdk", limit=3).fetch()
        print("Fetched", len(live_items), "public releases:")
        for item in live_items:
            print(f"  {item.item_id:<14} {item.title}")
        print("Same UpdateItem contract as the cached source, so nothing downstream changes.")
    except Exception as exc:                 # noqa: BLE001 - network is optional here
        print("Live fetch failed:", type(exc).__name__, exc)
        print("Falling back to the cached source; the lesson is unchanged.")
else:
    print("Skipped. The cached fixture already exercises every guardrail.")

In [ ]:
# Live MODEL. This is the only path where a model actually writes the update text.
from mini_harness import OpenRouterWebsiteProposer

live_proposer = proposer          # default: the deterministic copy-the-summary proposer
if LIVE:
    try:
        live_proposer = OpenRouterWebsiteProposer()
        sample = live_proposer(clean)
        print("Live proposal heading:", sample.heading)
        print("Live proposal body   :", sample.body[:200], "...")
        print()
        print("Compare with the deterministic version, which just copies the summary:")
        print("  ", deterministic_proposer(clean).body[:120], "...")
    except Exception as exc:                 # noqa: BLE001 - one failure must not stop the class
        print("Live proposer unavailable:", type(exc).__name__, exc)
        print("Falling back to deterministic_proposer.")
        live_proposer = proposer
else:
    print("MOCK mode: using deterministic_proposer, which copies the source summary verbatim.")
    print("No model is involved on this path - be honest about that when you present it.")

print()
print("Whichever proposer is used, the guardrails and the approval pause are identical.")
print("That is the point: the model writes text; Python decides what happens to it.")

## Step 10 — Where the scheduler fits

`run_website_agent.py` is the entry point a scheduler calls. It contains no business
logic: it finds new work, starts one bounded harness run, and stops at the approval.


In [ ]:
script = (PROJECT_ROOT / "run_website_agent.py").read_text(encoding="utf-8")
body = [line for line in script.splitlines() if line.startswith(("RUN_ROOT", "result =", "runtime ="))]
print("Key lines from run_website_agent.py:")
for line in body:
    print("  ", line)
print()
print("Run it from a terminal with:  python run_website_agent.py")
print("Schedule it with cron, Windows Task Scheduler or a CI schedule.")
print("The scheduler decides WHEN. It never decides WHETHER - policy and a human do.")

### Try it yourself

Predict: if you re-classified `publish_update` as `write` instead of `external`, what
would this capstone do differently?


In [ ]:
# --- Worked solution ---
# Risk levels are not labels; they are the control. Re-classify the tool and the
# entire human checkpoint disappears - so we do this on a THROWAWAY site folder.
from mini_harness import ToolRegistry, ToolSpec

throwaway_site = RUN_ROOT / "throwaway_site"
throwaway = build_website_registry(
    source, proposer, WebsiteGuardrails(throwaway_site, {"github.com"}),
    JSONStateStore(RUN_ROOT / "throwaway_state.json"))

# Rebuild the registry with publish_update downgraded from external to write.
downgraded = ToolRegistry()
for spec in throwaway.discover():
    risk = "write" if spec.name == "publish_update" else spec.risk
    downgraded.register(ToolSpec(spec.name, spec.description, spec.input_schema, risk),
                        throwaway.get(spec.name).function)

print("publish_update risk is now:", downgraded.get("publish_update").spec.risk)
print("Policy decision            :", decide(config, downgraded.get("publish_update").spec))
print()

runaway = HarnessRuntime(downgraded, MockModel(), EventStore())
outcome = runaway.run(config, clean.item_id)
print("Run status:", outcome.status, "  <- it completed; nobody was asked")
print("Throwaway site written    :", (throwaway_site / "content" / "updates.md").exists())
print()
print("One word in one ToolSpec removed the human from the loop entirely.")
print("Classifying a tool's risk honestly is the single highest-leverage safety decision")
print("in this whole harness - and it is a decision only a person can make.")

## Required live observation

Choose one bounded live observation: fetch up to three public releases or obtain one OpenRouter update proposal. Stop before approval. The cached source and captured trace are the outage fallback.


### Checkpoint

**1. The scheduler runs this every day at 06:00. Does that make it agentic?**

<details><summary>Show answer</summary>

No. The schedule is plain automation - a timer starting a process. The agentic part is deciding that a particular source item is new and worth a website change, and proposing a specific patch for it. Policy then decides whether that proposal may proceed, and in this design it always pauses before publishing.

</details>

**2. In mock mode the proposal text is copied from the source. Is the capstone still real?**

<details><summary>Show answer</summary>

The workflow is real: real files, real durable state, real guardrails, real policy and a real approval gate. What is not real is the *writing* - `deterministic_proposer` copies the summary and makes no model call. Step 9 swaps in `OpenRouterWebsiteProposer`, and note that not one guardrail or policy decision changes when it does.

</details>

### Recap

- Limitation: a workflow that touches real files and a real website is exactly where 'the model decided to' stops being an acceptable explanation.
- Layer added: the workflow's two actions registered as harness tools with honest risk levels, driven by the runtime, gated by policy, recorded as events and paused on a checkpoint - with domain guardrails inside each tool.
- Evidence: `propose_update` allowed and `publish_update` paused; the website file appeared only after an explicit approval; a poisoned source was refused before drafting; and downgrading one risk level removed the human gate entirely.


---

### Section 5.9 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


## Day 5 completion checklist

- [ ] I can explain how every section contributes to the **Mini AI Harness + Website Maintenance Agent**.
- [ ] I ran the deterministic path and at least one required live observation or classroom fallback.
- [ ] I attempted the pivotal exercise before reading its reference solution, and I can explain the solution line by line.
- [ ] I can identify the system state, safety boundary, and evidence used to judge the result.
